> **LangChain 1.x (2026)** — built on `langchain-core==1.2.30`, `langchain==1.0.0`. See `UPDATE_2026.md`.

# Chapter 6 — Assay Curation (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare/blob/main/notebooks/Chapter%2006.%20LangChain%20for%20Chemistry/LC4LSH_Chapter_6_Assay_Curation.ipynb)

**Learning objectives**
- Normalize units and relations (<, >, =) in assay data
- Record endpoint context and transformations
- Detect duplicate/conflicting measurements
- Export a curated, documented table

> Runtime: ~5 min (local, no API)  
> Cost: free  
> Data: small synthetic assay set


## Environment setup


### Secrets (optional LLM only)


In [ ]:
import os
try:
    from google.colab import userdata  # type: ignore
    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False
if not IN_COLAB:
    try:
        from dotenv import load_dotenv  # type: ignore
        load_dotenv()
    except Exception:
        pass

def get_secret(name, default=None):
    if IN_COLAB and userdata is not None:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    return os.getenv(name, default)

# These notebooks run locally on RDKit; an LLM is OPTIONAL for narrative only.
API_KEY_PROVIDER = "OPENAI"  # "GEMINI" | "OPENAI" | "GROQ" | "ANTHROPIC"
if API_KEY_PROVIDER == "OPENAI":
    os.environ["OPENAI_API_KEY"] = get_secret("LC4LS_OPENAI_API_KEY", "sk-...")
print("Optional LLM provider:", API_KEY_PROVIDER, "(RDKit runs without it)")
os.environ["HF_TOKEN"] = get_secret("HF_TOKEN", "") or ""


### Install pinned dependencies


In [ ]:
%pip install -q "rdkit==2023.9.6" "langchain==1.0.0" "langchain-core==1.2.30" "langchain-openai==1.0.0" "pandas>=2.0" "matplotlib>=3.8" "scipy>=1.11" python-dotenv
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)


In [ ]:
LANGSMITH_API_KEY = get_secret("LANGSMITH_API_KEY", "")
LANGSMITH_PROJECT = "lc4lsh-chapter6-assay-curation"
if LANGSMITH_API_KEY and LANGSMITH_API_KEY.startswith(("lsv2_", "ls__")):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    print("LangSmith ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith OFF (fine — these notebooks are local/RDKit-first)")


## Why curate assays?

Raw bioactivity data mixes units (nM/µM), relations (`<`, `>`, `=`), and endpoints (IC50/Ki/EC50). Modeling on uncurated data silently mixes apples and oranges. Curation makes every value **explicit and traceable**.


## 1. Load a raw assay table


In [ ]:
import pandas as pd

raw = pd.DataFrame({
    "compound": ["A", "B", "C", "D", "E", "F"],
    "endpoint": ["IC50", "IC50", "Ki", "IC50", "EC50", "IC50"],
    "relation": ["=", "<", "=", ">", "=", "="],
    "value":    [10.0, 5.0, 200.0, 1000.0, 50.0, 10.0],
    "unit":     ["nM", "nM", "nM", "nM", "uM", "nM"],
})
print(raw)


## 2. Normalize units to a single scale (nM)


In [ ]:
UNIT_TO_NM = {"nM": 1.0, "uM": 1000.0, "µM": 1000.0, "pM": 0.001, "mM": 1e6}

raw["value_nM"] = raw.apply(lambda r: r["value"] * UNIT_TO_NM[r["unit"]], axis=1)
raw["transform"] = raw.apply(lambda r: f'{r["value"]}{r["unit"]} -> {r["value_nM"]}nM', axis=1)
print(raw[["compound", "transform"]])


## 3. Make relations explicit (censoring)


In [ ]:
def censor(row):
    if row["relation"] == "<":
        return f'upper_bound={row["value_nM"]}nM (true value below)'
    if row["relation"] == ">":
        return f'lower_bound={row["value_nM"]}nM (true value above)'
    return f'exact={row["value_nM"]}nM'

raw["interpretation"] = raw.apply(censor, axis=1)
print(raw[["compound", "relation", "interpretation"]])


## 4. Endpoint context & comparability


In [ ]:
ENDPOINT_KIND = {"IC50": "functional inhibition", "Ki": "binding affinity",
                 "EC50": "functional activation", "IC90": "functional inhibition (90%)"}
raw["endpoint_kind"] = raw["endpoint"].map(ENDPOINT_KIND)
print(raw[["endpoint", "endpoint_kind"]].drop_duplicates())
print("NOTE: do not pool Ki (binding) with IC50 (functional) without justification.")


## 5. Duplicate / conflicting measurements


In [ ]:
dup = raw[raw.duplicated(["compound", "endpoint"], keep=False)]
conflicts = dup.groupby(["compound", "endpoint"])["value_nM"].agg(["min", "max", "count"])
conflicts["fold_range"] = conflicts["max"] / conflicts["min"]
print("Repeated compound+endpoint measurements:")
print(conflicts)


## 6. Export curated table


In [ ]:
curated = raw[["compound", "endpoint", "endpoint_kind", "relation", "value_nM", "interpretation", "transform"]]
curated.to_csv("curated_assay.csv", index=False)
print(curated.to_string(index=False))
print("Wrote curated_assay.csv")


## Limitations & safety notes

- Unit conversion assumes the `UNIT_TO_NM` map; extend/validate for your data.
- `<`/`>` values are censored, not exact.
- Pooling different endpoints (Ki vs IC50) needs explicit scientific justification.
- Local/free; no API needed.


In [ ]:
# Cleanup
import gc
for _v in ("mol", "mols", "df", "llm", "model", "img", "raw", "curated"):
    globals().pop(_v, None)
try:
    import torch
    torch.cuda.empty_cache()
except Exception:
    pass
gc.collect()
print("Cleanup complete.")


## Exercises

<details><summary>Why keep the relation (<, >, =)?</summary>It marks censored data; dropping it turns bounds into fake exact values and biases models.</details>

<details><summary>Why not pool Ki and IC50?</summary>They measure different things (binding vs functional response); combining them adds noise/error.</details>

<details><summary>Why log transformations?</summary>So every normalized value is traceable back to its raw value and unit.</details>

### Tasks
- **Task A** - Add a `pchembl` (-log10 molar) column computed from `value_nM`.
- **Task B** - Flag measurements with fold_range > 3 as conflicts needing review.
- **Task C** - Add a units-validation step that errors on an unknown unit string.
- **Task D** - Export a JSON manifest recording unit map, relation policy, and row counts.
